# Procesamiento y validación de los Conteos Censales de Participación Ciudadana 2018

**Proyecto:** Hackea la participación: Datos que construyen democracia  
**Reto 1:** Perfiles ciudadanos de participación  
**Etapa:** Homologación histórica 2009–2024  
**Año procesado:** 2018

## Objetivo

Este notebook procesa la base oficial de los **Conteos Censales de Participación Ciudadana 2018** del Instituto Nacional Electoral (INE) y la transforma a la misma unidad de análisis utilizada para 2024:

\[
\text{Entidad} + \text{Grupo de edad} + \text{Sexo} + \text{Tipo de sección}
\]

El propósito es obtener una base agregada y validada que pueda integrarse posteriormente con los demás años para construir el historial de participación utilizado en la segmentación de perfiles ciudadanos.

### Principios de trabajo

- Se conservan los datos originales sin modificación.
- Se documentan diferencias de estructura respecto de 2024.
- La participación se calcula **después de agregar los conteos**, no como promedio simple de porcentajes.
- Los grupos pequeños se conservan y únicamente se marcan para análisis de sensibilidad.
- Si una variable esperada no existe o tiene una codificación distinta, el notebook lo reporta antes de continuar.

## Fuente oficial

La base 2018 forma parte de los Conteos Censales de Participación Ciudadana publicados por el INE.  
La descarga utilizada corresponde al archivo oficial:

`DatosAbiertos_DECEyEC_ConteosCensales2018.zip`

El archivo ZIP se carga manualmente en Google Colab para evitar respuestas HTML o descargas incompletas del servidor.

> **Nota de reproducibilidad:** los archivos descargados o subidos directamente a Colab son temporales y desaparecen al reiniciar el entorno.

In [5]:
import os
import io
import re
import zipfile
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

## 1. Carga del archivo ZIP

Descarga desde la página oficial del INE el archivo correspondiente a **2018** y súbelo a Google Colab.

El archivo oficial se identifica como:

`DatosAbiertos_DECEyEC_ConteosCensales2018.zip`

Para evitar que Colab seleccione por accidente el ZIP de otro año, se recomienda trabajar en una sesión limpia o eliminar previamente otros archivos `.zip` de `/content/`.

> **Nota:** los archivos cargados directamente a Colab son temporales y desaparecen al reiniciar el entorno.

In [6]:
import os
import io
import re
import zipfile
import requests
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

URL_2018 = "https://www.ine.mx/wp-content/uploads/2022/04/DatosAbiertos_DECEyEC_ConteosCensales2018.zip"
RUTA_ZIP = "/content/DatosAbiertos_DECEyEC_ConteosCensales2021.zip"

if not os.path.exists(RUTA_ZIP):
    print("El ZIP no está en /content/. Intentando descarga desde el INE...")
    try:
        r = requests.get(URL_2018, timeout=90)
        r.raise_for_status()
        with open(RUTA_ZIP, "wb") as f:
            f.write(r.content)
        print("Descarga completada:", RUTA_ZIP)
    except Exception as e:
        print("No fue posible descargar automáticamente el archivo.")
        print("Motivo:", e)
        print("\nSube manualmente DatosAbiertos_DECEyEC_ConteosCensales2018.zip")

        try:
            from google.colab import files
            subidos = files.upload()
            nombres = list(subidos.keys())
            if not nombres:
                raise RuntimeError("No se cargó ningún archivo.")

            nombre_subido = nombres[0]
            with open(RUTA_ZIP, "wb") as f:
                f.write(subidos[nombre_subido])

            print("Archivo guardado como:", RUTA_ZIP)
        except ImportError:
            raise RuntimeError(
                "Este bloque está pensado para Google Colab. "
                "Coloca manualmente el ZIP en /content/."
            )

print("ZIP listo:", os.path.exists(RUTA_ZIP))

ZIP listo: True


## 2. Inventario del ZIP

Se identifican los archivos CSV disponibles y, cuando existe, el archivo de descripción de campos incluido por el INE.

In [7]:
with zipfile.ZipFile(RUTA_ZIP, "r") as z:
    archivos = z.namelist()

archivos_csv = [a for a in archivos if a.lower().endswith(".csv")]
archivos_txt = [a for a in archivos if a.lower().endswith(".txt")]

print("Archivos totales:", len(archivos))
print("Archivos CSV:", len(archivos_csv))
print("Archivos TXT:", len(archivos_txt))

print("\nPrimeros archivos:")
for a in archivos[:10]:
    print(" -", a)

Archivos totales: 34
Archivos CSV: 32
Archivos TXT: 1

Primeros archivos:
 - DatosAbiertos_DECEyEC_ConteosCensales2018/
 - DatosAbiertos_DECEyEC_ConteosCensales2018/01_descripcioncampos_conteoscensales2018.txt
 - DatosAbiertos_DECEyEC_ConteosCensales2018/datosabiertos_deceyec_conteoscensales2018_ags.csv
 - DatosAbiertos_DECEyEC_ConteosCensales2018/datosabiertos_deceyec_conteoscensales2018_bc.csv
 - DatosAbiertos_DECEyEC_ConteosCensales2018/datosabiertos_deceyec_conteoscensales2018_bcs.csv
 - DatosAbiertos_DECEyEC_ConteosCensales2018/datosabiertos_deceyec_conteoscensales2018_camp.csv
 - DatosAbiertos_DECEyEC_ConteosCensales2018/datosabiertos_deceyec_conteoscensales2018_cdmx.csv
 - DatosAbiertos_DECEyEC_ConteosCensales2018/datosabiertos_deceyec_conteoscensales2018_chih.csv
 - DatosAbiertos_DECEyEC_ConteosCensales2018/datosabiertos_deceyec_conteoscensales2018_chis.csv
 - DatosAbiertos_DECEyEC_ConteosCensales2018/datosabiertos_deceyec_conteoscensales2018_coah.csv


### Archivo de descripción

Cuando el ZIP contiene un archivo `.txt`, se muestran sus primeras líneas.  
Este archivo debe considerarse la referencia principal para interpretar la codificación específica de 2018.

In [8]:
if archivos_txt:
    with zipfile.ZipFile(RUTA_ZIP, "r") as z:
        contenido = z.read(archivos_txt[0])

    texto = None
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            texto = contenido.decode(enc)
            break
        except UnicodeDecodeError:
            pass

    print("\n".join(texto.splitlines()[:80]))
else:
    print("No se encontró archivo TXT de descripción dentro del ZIP.")

Descripción archivo
AELEC	Año de elección (2009,2012,2015, 2018 y 2021)
FELECCION	Fecha de la jornada electoral (formato corto)
EDOCVE	Clave INE de entidad federativa
EDONOM	Nombre de la entidad federativa
MPIOCVE	Clave INE de Municipio
MPIONOM	Nombre del municipio
SECCION	Sección electoral en el año correspondiente
TIPOSEC	Tipo de sección en el año correspondiente (Urbana, Mixta, Rural)
DEF	Distrito Electoral Federal en el marco cartográfico de la DERFE 2021
DEL	Distrito Electoral Local en el marco cartográfico de la DERFE 2021
SEXO	Sexo de las personas (0 Hombres; 1 Mujeres)
EDAD	Años de edad cumplidos al día de la elección de las personas
LN	Lista nominal de los cuadernillos para todas las casillas catalogadas
SV	Registros con marca de voto.
NV	Registros sin marca de voto.
NS	Registros no especificados en participación, pues corresponden a:
	- Registro correspondiente a un cuadernillo sin marcas de votación.
	- Registro correspondiente a un cuadernillo no disponible para su captura.

## 3. Función de lectura robusta

Algunas bases históricas pueden utilizar codificaciones de texto distintas.  
La siguiente función intenta leer cada CSV con varias codificaciones comunes, sin alterar los datos.

In [9]:
def leer_csv_desde_zip(z, nombre):
    ultimo_error = None

    for encoding in ["utf-8", "latin-1", "cp1252"]:
        try:
            with z.open(nombre) as f:
                return pd.read_csv(f, encoding=encoding)
        except UnicodeDecodeError as e:
            ultimo_error = e

    raise ultimo_error

## 4. Inspección de un archivo estatal

Antes de procesar todo el país se revisa un CSV para conocer las columnas y algunos valores de ejemplo.

In [10]:
with zipfile.ZipFile(RUTA_ZIP, "r") as z:
    df_muestra = leer_csv_desde_zip(z, archivos_csv[0])

print("Archivo de muestra:", archivos_csv[0])
print("Dimensiones:", df_muestra.shape)
display(df_muestra.head())
print("\nColumnas:")
print(df_muestra.columns.tolist())

Archivo de muestra: DatosAbiertos_DECEyEC_ConteosCensales2018/datosabiertos_deceyec_conteoscensales2018_ags.csv
Dimensiones: (82051, 16)


,AELEC,FELECCION,EDOCVE,EDONOM,MPIOCVE,MPIONOM,SECCION,TIPOSEC,DEF,DEL,SEXO,EDAD,LN,SV,NV,NS
0,2018,1/7/2018,1,AGUASCALIENTES,1,Aguascalientes,1,U,3,6,0,18,24,18,6,0
1,2018,1/7/2018,1,AGUASCALIENTES,1,Aguascalientes,1,U,3,6,0,19,23,20,3,0
2,2018,1/7/2018,1,AGUASCALIENTES,1,Aguascalientes,1,U,3,6,0,20,23,19,4,0
3,2018,1/7/2018,1,AGUASCALIENTES,1,Aguascalientes,1,U,3,6,0,21,19,12,7,0
4,2018,1/7/2018,1,AGUASCALIENTES,1,Aguascalientes,1,U,3,6,0,22,19,10,9,0



Columnas:
['AELEC', 'FELECCION', 'EDOCVE', 'EDONOM', 'MPIOCVE', 'MPIONOM', 'SECCION', 'TIPOSEC', 'DEF', 'DEL', 'SEXO', 'EDAD', 'LN', 'SV', 'NV', 'NS']


## 5. Compatibilidad estructural con el procesamiento 2024

Para mantener una metodología común entre años, se revisan las variables necesarias para la unidad de análisis y los indicadores.

Las variables mínimas esperadas son:

- `EDONOM`: entidad federativa.
- `EDAD`: edad.
- `SEXO`: sexo.
- `TIPOSEC`: tipo de sección.
- `LN`: lista nominal.
- `SV`: sí votaron.
- `NV`: no votaron.

`NS` se utiliza cuando existe para representar registros con situación de participación no especificada.

> Si `NS` no está presente, el notebook solo asumirá `NS = 0` cuando pueda comprobarse que `LN = SV + NV` en todos los registros.

In [11]:
columnas_esperadas = ["EDONOM", "EDAD", "SEXO", "TIPOSEC", "LN", "SV", "NV", "NS"]

compatibilidad = pd.DataFrame({
    "VARIABLE": columnas_esperadas,
    "PRESENTE_2018": [c in df_muestra.columns for c in columnas_esperadas]
})

display(compatibilidad)

obligatorias = ["EDONOM", "EDAD", "SEXO", "TIPOSEC", "LN", "SV", "NV"]
faltantes = [c for c in obligatorias if c not in df_muestra.columns]

if faltantes:
    raise ValueError(
        "Faltan columnas obligatorias para aplicar el pipeline: "
        + ", ".join(faltantes)
    )

print("Variables obligatorias disponibles.")

,VARIABLE,PRESENTE_2018
0,EDONOM,True
1,EDAD,True
2,SEXO,True
3,TIPOSEC,True
4,LN,True
5,SV,True
6,NV,True
7,NS,True


Variables obligatorias disponibles.


## 6. Preparación metodológica

Se utilizan los mismos grupos de edad definidos para 2024:

- 18–19
- 20–24
- 25–29
- ...
- 70–74
- 75+

El grupo `75+` evita que edades muy altas generen categorías con muy pocos registros y mantiene una clasificación interpretable.

La codificación descriptiva de `SEXO` y `TIPOSEC` se aplica únicamente a los códigos reconocidos.  
Cualquier código adicional queda identificado para revisión y no se reemplaza silenciosamente.

In [12]:
BINS_EDAD = [
    18, 20, 25, 30, 35, 40, 45,
    50, 55, 60, 65, 70, 75, float("inf")
]

LABELS_EDAD = [
    "18-19", "20-24", "25-29", "30-34",
    "35-39", "40-44", "45-49", "50-54",
    "55-59", "60-64", "65-69", "70-74", "75+"
]

MAP_SEXO = {
    0: "Hombre",
    1: "Mujer",
    2: "No binario"
}

MAP_TIPOSEC = {
    "U": "Urbana",
    "M": "Mixta",
    "R": "Rural"
}

## 7. Función de procesamiento por entidad

Para reducir el uso de memoria, cada archivo estatal se prepara y agrega antes de integrarse a la base nacional.

Este procedimiento produce el mismo tipo de unidad de análisis que en 2024, pero evita mantener simultáneamente millones de filas crudas en memoria.

In [13]:
def preparar_y_agrupar(df, nombre_archivo=""):
    df = df.copy()

    # Convertir variables cuantitativas a numéricas.
    for col in ["EDAD", "LN", "SV", "NV"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Manejo responsable de NS.
    if "NS" not in df.columns:
        prueba = (
            df["LN"].notna()
            & df["SV"].notna()
            & df["NV"].notna()
            & (df["LN"] == df["SV"] + df["NV"])
        )

        if prueba.all():
            df["NS"] = 0
            ns_inferido = True
        else:
            raise ValueError(
                f"{nombre_archivo}: no existe NS y LN != SV + NV en algunos registros. "
                "Debe revisarse el diccionario de variables antes de continuar."
            )
    else:
        df["NS"] = pd.to_numeric(df["NS"], errors="coerce")
        ns_inferido = False

    # Revisar valores no numéricos en conteos.
    if df[["EDAD", "LN", "SV", "NV", "NS"]].isnull().any().any():
        filas_problematicas = df[
            df[["EDAD", "LN", "SV", "NV", "NS"]].isnull().any(axis=1)
        ]
        raise ValueError(
            f"{nombre_archivo}: existen valores no numéricos o faltantes "
            f"en variables esenciales ({len(filas_problematicas)} filas)."
        )

    # Grupos de edad.
    df["GRUPO_EDAD"] = pd.cut(
        df["EDAD"],
        bins=BINS_EDAD,
        labels=LABELS_EDAD,
        right=False
    )

    if df["GRUPO_EDAD"].isna().any():
        fuera_rango = df.loc[df["GRUPO_EDAD"].isna(), "EDAD"].unique()
        raise ValueError(
            f"{nombre_archivo}: hay edades fuera del esquema definido: {fuera_rango[:20]}"
        )

    # Descripciones.
    df["SEXO_DESC"] = df["SEXO"].map(MAP_SEXO)
    df["TIPOSEC_DESC"] = df["TIPOSEC"].map(MAP_TIPOSEC)

    # Detectar códigos no contemplados.
    codigos_sexo_desconocidos = sorted(
        df.loc[df["SEXO_DESC"].isna(), "SEXO"].dropna().unique().tolist()
    )
    codigos_tipo_desconocidos = sorted(
        df.loc[df["TIPOSEC_DESC"].isna(), "TIPOSEC"].dropna().unique().tolist()
    )

    if codigos_sexo_desconocidos:
        raise ValueError(
            f"{nombre_archivo}: códigos de SEXO no reconocidos: "
            f"{codigos_sexo_desconocidos}. Revisar diccionario 2018."
        )

    if codigos_tipo_desconocidos:
        raise ValueError(
            f"{nombre_archivo}: códigos de TIPOSEC no reconocidos: "
            f"{codigos_tipo_desconocidos}. Revisar diccionario 2018."
        )

    # Totales originales para validación.
    totales_originales = df[["LN", "SV", "NV", "NS"]].sum()

    # Comprobar consistencia LN = SV + NV + NS.
    consistencia = df["LN"] == (df["SV"] + df["NV"] + df["NS"])
    registros_inconsistentes = int((~consistencia).sum())

    # Agregación.
    agrupado = (
        df.groupby(
            ["EDONOM", "GRUPO_EDAD", "SEXO_DESC", "TIPOSEC_DESC"],
            observed=True,
            as_index=False
        )[["LN", "SV", "NV", "NS"]]
        .sum()
    )

    return agrupado, totales_originales, registros_inconsistentes, ns_inferido

## 8. Procesamiento de todos los archivos 2018

Cada CSV se procesa con el mismo procedimiento y posteriormente se integran los resultados nacionales.

In [14]:
agrupados = []
totales_originales_2018 = pd.Series({"LN": 0, "SV": 0, "NV": 0, "NS": 0}, dtype="int64")
inconsistencias_totales = 0
archivos_con_ns_inferido = []

with zipfile.ZipFile(RUTA_ZIP, "r") as z:
    for i, nombre in enumerate(archivos_csv, start=1):
        df_estado = leer_csv_desde_zip(z, nombre)

        agrupado_estado, totales_estado, inconsistencias, ns_inferido = preparar_y_agrupar(
            df_estado,
            nombre_archivo=nombre
        )

        agrupados.append(agrupado_estado)
        totales_originales_2018 = totales_originales_2018.add(totales_estado, fill_value=0)
        inconsistencias_totales += inconsistencias

        if ns_inferido:
            archivos_con_ns_inferido.append(nombre)

        print(f"[{i}/{len(archivos_csv)}] Procesado: {nombre}")

df_2018_agrupado = (
    pd.concat(agrupados, ignore_index=True)
    .groupby(
        ["EDONOM", "GRUPO_EDAD", "SEXO_DESC", "TIPOSEC_DESC"],
        observed=True,
        as_index=False
    )[["LN", "SV", "NV", "NS"]]
    .sum()
)

print("\nProcesamiento nacional terminado.")
print("Dimensiones de la base agregada:", df_2018_agrupado.shape)
print("Entidades:", df_2018_agrupado["EDONOM"].nunique())
print("Registros inconsistentes LN = SV + NV + NS:", inconsistencias_totales)
print("Archivos en los que NS tuvo que inferirse:", len(archivos_con_ns_inferido))

[1/32] Procesado: DatosAbiertos_DECEyEC_ConteosCensales2018/datosabiertos_deceyec_conteoscensales2018_ags.csv
[2/32] Procesado: DatosAbiertos_DECEyEC_ConteosCensales2018/datosabiertos_deceyec_conteoscensales2018_bc.csv
[3/32] Procesado: DatosAbiertos_DECEyEC_ConteosCensales2018/datosabiertos_deceyec_conteoscensales2018_bcs.csv
[4/32] Procesado: DatosAbiertos_DECEyEC_ConteosCensales2018/datosabiertos_deceyec_conteoscensales2018_camp.csv
[5/32] Procesado: DatosAbiertos_DECEyEC_ConteosCensales2018/datosabiertos_deceyec_conteoscensales2018_cdmx.csv
[6/32] Procesado: DatosAbiertos_DECEyEC_ConteosCensales2018/datosabiertos_deceyec_conteoscensales2018_chih.csv
[7/32] Procesado: DatosAbiertos_DECEyEC_ConteosCensales2018/datosabiertos_deceyec_conteoscensales2018_chis.csv
[8/32] Procesado: DatosAbiertos_DECEyEC_ConteosCensales2018/datosabiertos_deceyec_conteoscensales2018_coah.csv
[9/32] Procesado: DatosAbiertos_DECEyEC_ConteosCensales2018/datosabiertos_deceyec_conteoscensales2018_col.csv
[10/32

## 9. Cálculo de indicadores

La participación se calcula sobre los registros con resultado determinado:

\[
	ext{Participación} = rac{SV}{SV+NV}	imes 100
\]

\[
	ext{Abstención} = rac{NV}{SV+NV}	imes 100
\]

Los registros `NS` no se incorporan al denominador de estas dos tasas.  
También se calcula su proporción respecto de la lista nominal para conservar un indicador de calidad de la información.

In [15]:
df_2018_agrupado["N_DETERMINADO"] = (
    df_2018_agrupado["SV"] + df_2018_agrupado["NV"]
)

den = df_2018_agrupado["N_DETERMINADO"]

df_2018_agrupado["PARTICIPACION_PCT"] = np.where(
    den > 0,
    df_2018_agrupado["SV"] / den * 100,
    np.nan
)

df_2018_agrupado["ABSTENCION_PCT"] = np.where(
    den > 0,
    df_2018_agrupado["NV"] / den * 100,
    np.nan
)

df_2018_agrupado["NO_ESPECIFICADO_PCT"] = np.where(
    df_2018_agrupado["LN"] > 0,
    df_2018_agrupado["NS"] / df_2018_agrupado["LN"] * 100,
    np.nan
)

# Misma bandera provisional utilizada en 2024.
df_2018_agrupado["GRUPO_PEQUENO"] = (
    df_2018_agrupado["N_DETERMINADO"] < 100
)

display(df_2018_agrupado.head(20))

,EDONOM,GRUPO_EDAD,SEXO_DESC,TIPOSEC_DESC,LN,SV,NV,NS,N_DETERMINADO,PARTICIPACION_PCT,ABSTENCION_PCT,NO_ESPECIFICADO_PCT,GRUPO_PEQUENO
0,AGUASCALIENTES,18-19,Hombre,Mixta,5882,2734,3030,118,5764,47.432339,52.567661,2.006120,False
1,AGUASCALIENTES,18-19,Hombre,Rural,2649,1336,1281,32,2617,51.050822,48.949178,1.208003,False
2,AGUASCALIENTES,18-19,Hombre,Urbana,14588,7727,6694,167,14421,53.581582,46.418418,1.144777,False
3,AGUASCALIENTES,18-19,Mujer,Mixta,6047,3293,2630,124,5923,55.596826,44.403174,2.050604,False
4,AGUASCALIENTES,18-19,Mujer,Rural,2587,1557,993,37,2550,61.058824,38.941176,1.430228,False
5,AGUASCALIENTES,18-19,Mujer,Urbana,14584,8440,5979,165,14419,58.533879,41.466121,1.131377,False
6,AGUASCALIENTES,20-24,Hombre,Mixta,15282,5881,9110,291,14991,39.230205,60.769795,1.904201,False
7,AGUASCALIENTES,20-24,Hombre,Rural,7465,2923,4433,109,7356,39.736270,60.263730,1.460147,False
8,AGUASCALIENTES,20-24,Hombre,Urbana,41100,18841,21697,562,40538,46.477379,53.522621,1.367397,False
9,AGUASCALIENTES,20-24,Mujer,Mixta,15334,7543,7505,286,15048,50.126263,49.873737,1.865136,False


## 10. Validaciones de integridad

Las siguientes pruebas deben cumplirse antes de usar la base para el análisis histórico:

1. La agregación debe conservar los totales de `LN`, `SV`, `NV` y `NS`.
2. Deben aparecer las entidades disponibles en los archivos oficiales.
3. Participación + abstención debe ser aproximadamente 100 % cuando el denominador es mayor que cero.
4. Los grupos pequeños se conservan; no se eliminan en esta etapa.

In [16]:
totales_agrupados_2018 = df_2018_agrupado[["LN", "SV", "NV", "NS"]].sum()

print("Totales originales:")
display(totales_originales_2018)

print("\nTotales después de agrupar:")
display(totales_agrupados_2018)

conserva_totales = np.allclose(
    totales_originales_2018.astype(float).values,
    totales_agrupados_2018.astype(float).values
)

print("\n¿Se conservaron los totales?:", conserva_totales)

Totales originales:


,0
LN,89123997
SV,51784399
NV,31249856
NS,6089742



Totales después de agrupar:


,0
LN,89123997
SV,51784399
NV,31249856
NS,6089742



¿Se conservaron los totales?: True


In [17]:
suma_tasas = (
    df_2018_agrupado["PARTICIPACION_PCT"]
    + df_2018_agrupado["ABSTENCION_PCT"]
)

print("Resumen de PARTICIPACION_PCT:")
display(df_2018_agrupado["PARTICIPACION_PCT"].describe())

print("\nParticipación + abstención:")
display(suma_tasas.describe())

print("\nGrupos pequeños:")
display(df_2018_agrupado["GRUPO_PEQUENO"].value_counts())

Resumen de PARTICIPACION_PCT:


,PARTICIPACION_PCT
count,2496.000000
mean,63.803568
std,11.055101
min,27.961297
25%,55.953979
50%,64.601567
75%,72.366948
max,93.000838



Participación + abstención:


,0
count,2.496000e+03
mean,1.000000e+02
std,3.168077e-15
min,1.000000e+02
25%,1.000000e+02
50%,1.000000e+02
75%,1.000000e+02
max,1.000000e+02



Grupos pequeños:


,count
GRUPO_PEQUENO,
False,2472
True,24


## 11. Resumen nacional descriptivo

El siguiente bloque calcula la participación nacional a partir de los **totales de conteos**, por lo que no debe confundirse con el promedio simple de los porcentajes de los grupos.

In [18]:
totales = df_2018_agrupado[["LN", "SV", "NV", "NS"]].sum()
n_determinado = totales["SV"] + totales["NV"]

resumen_nacional_2018 = pd.Series({
    "LN": totales["LN"],
    "SV": totales["SV"],
    "NV": totales["NV"],
    "NS": totales["NS"],
    "N_DETERMINADO": n_determinado,
    "PARTICIPACION_PCT_NACIONAL": (
        totales["SV"] / n_determinado * 100
        if n_determinado > 0 else np.nan
    ),
    "ABSTENCION_PCT_NACIONAL": (
        totales["NV"] / n_determinado * 100
        if n_determinado > 0 else np.nan
    ),
    "NO_ESPECIFICADO_PCT_NACIONAL": (
        totales["NS"] / totales["LN"] * 100
        if totales["LN"] > 0 else np.nan
    )
})

display(resumen_nacional_2018)

,0
LN,8.912400e+07
SV,5.178440e+07
NV,3.124986e+07
NS,6.089742e+06
N_DETERMINADO,8.303426e+07
PARTICIPACION_PCT_NACIONAL,6.236510e+01
ABSTENCION_PCT_NACIONAL,3.763490e+01
NO_ESPECIFICADO_PCT_NACIONAL,6.832887e+00


## 12. Tabla de compatibilidad para la integración histórica

Esta tabla registra qué variables del esquema común están disponibles en 2018.  
Más adelante se combinará con las revisiones de 2009, 2012, 2015, 2018 y 2024.

In [19]:
compatibilidad["OBSERVACION"] = ""

if "NS" not in df_muestra.columns:
    compatibilidad.loc[
        compatibilidad["VARIABLE"] == "NS",
        "OBSERVACION"
    ] = "No aparece en la muestra; el pipeline verifica si puede inferirse como 0."

compatibilidad["ANIO"] = 2018
compatibilidad = compatibilidad[["ANIO", "VARIABLE", "PRESENTE_2018", "OBSERVACION"]]

display(compatibilidad)

,ANIO,VARIABLE,PRESENTE_2018,OBSERVACION
0,2018,EDONOM,True,
1,2018,EDAD,True,
2,2018,SEXO,True,
3,2018,TIPOSEC,True,
4,2018,LN,True,
5,2018,SV,True,
6,2018,NV,True,
7,2018,NS,True,


## 13. Exportación

Se generan dos archivos:

1. `conteos_censales_2018_agrupado.csv`: base procesada para la integración histórica.
2. `compatibilidad_variables_2018.csv`: registro de variables disponibles para documentar la homologación.

In [20]:
SALIDA_DATOS = "/content/conteos_censales_2018_agrupado.csv"
SALIDA_COMPAT = "/content/compatibilidad_variables_2018.csv"

df_2018_agrupado.to_csv(
    SALIDA_DATOS,
    index=False,
    encoding="utf-8-sig"
)

compatibilidad.to_csv(
    SALIDA_COMPAT,
    index=False,
    encoding="utf-8-sig"
)

print("Archivos generados:")
print(" -", SALIDA_DATOS)
print(" -", SALIDA_COMPAT)

Archivos generados:
 - /content/conteos_censales_2018_agrupado.csv
 - /content/compatibilidad_variables_2018.csv


### Descarga opcional en Google Colab

Ejecuta la siguiente celda si quieres descargar inmediatamente los dos CSV a tu computadora.

In [22]:
# Descomenta estas líneas si deseas descargar los archivos:
#
# from google.colab import files
# files.download(SALIDA_DATOS)
# files.download(SALIDA_COMPAT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Notas para el documento técnico

### Metodología aplicada a 2018

La base de los Conteos Censales de Participación Ciudadana 2018 se procesó mediante un procedimiento homogéneo con el utilizado para 2024. Los registros se agruparon por entidad federativa, grupo de edad, sexo y tipo de sección electoral. Antes de calcular los indicadores de participación se sumaron los conteos de lista nominal, personas que sí votaron, personas que no votaron y, cuando estuvo disponible, registros de participación no especificada.

La tasa de participación se calculó como:

\[
\frac{SV}{SV+NV}\times100
\]

mientras que la abstención se calculó como:

\[
\frac{NV}{SV+NV}\times100
\]

Los registros con tamaño determinado menor a 100 se conservaron y se identificaron mediante la variable `GRUPO_PEQUENO`, con el objetivo de evaluar posteriormente la sensibilidad del método de segmentación ante grupos de tamaño reducido.

### Decisiones que deben mantenerse en los demás años

| Decisión | Criterio |
|---|---|
| Unidad de análisis | Entidad + grupo de edad + sexo + tipo de sección |
| Grupos de edad | Mismo esquema utilizado en 2024 |
| Porcentajes | Calcular después de sumar los conteos |
| `NS` | Excluir del denominador de participación/abstención y conservar como indicador separado |
| Grupos pequeños | No eliminar; marcar con `N_DETERMINADO < 100` |
| Códigos desconocidos | Detener el pipeline y revisar el diccionario, no recodificar automáticamente |

### Próximo paso

Una vez validado 2018, se aplicará el mismo pipeline a 2018, 2015, 2012 y 2009. Después se integrarán los años comparables para construir variables históricas de participación, tendencia y variabilidad antes de realizar el clustering.